## 面试问题

跨步指代一致性：前几步产出的实体/ID 怎么稳定引用？

## 回答主线

前面步骤产出的实体要在后续被稳定引用。自然语言指代（「那个订单」）在多实体时会指错或漂移；正确做法是给每个实体分配稳定 handle，维护 ref 表，后续用 handle 精确引用。本 Notebook 创建两个订单，对比自然语言指代（漂移到错误订单、发错人）与 handle 引用（精确命中）。

## 真实案例

循环先后创建订单 A(Alice) 和 B(Bob)，随后要给 A 发货。自然语言「第一个订单」可能被记混指向 B；`ref://order/a` 精确无误。数据为教学实体，不代表真实订单系统。

In [1]:
entities = []  # 循环产出的实体列表。
entities.append({"handle": "ref://order/a", "kind": "order", "customer": "Alice", "amount": 100})  # 第一步产出订单 A。
entities.append({"handle": "ref://order/b", "kind": "order", "customer": "Bob", "amount": 200})  # 第二步产出订单 B。

print("已产出实体数:", len(entities))  # 展示产出的实体数量。
for e in entities:  # 逐个打印实体。
    print("  ", e["handle"], e["customer"], e["amount"])  # 展示每个实体的 handle 与属性。

已产出实体数: 2
   ref://order/a Alice 100
   ref://order/b Bob 200


## 基线（Baseline）

反面基线：用自然语言短语指代实体。当同类实体多于一个时，「第一个订单」这种指代容易被模型记混，漂移到错误实体。

In [2]:
def resolve_by_language(entities, phrase):  # 用自然语言短语指代实体的脆弱方式。
    if phrase == "第一个订单":  # 模型可能把顺序记混。
        return entities[1]  # 模拟指代漂移到了订单 B。
    return entities[0]  # 兜底返回第一个。

target_by_language = resolve_by_language(entities, "第一个订单")  # 用自然语言指代目标订单。
print("自然语言指代目标:", target_by_language["handle"], target_by_language["customer"])  # 展示指代漂移到了 B。

自然语言指代目标: ref://order/b Bob


## 失败案例与修正

自然语言指代漂移会把货发给错误客户。修正是 handle 引用：构建 ref 表，用稳定 handle 精确解析，引用不存在的 handle 直接报错而非静默取错。

In [3]:
ref_table = {e["handle"]: e for e in entities}  # 构建 handle 到实体的引用表。

def resolve_by_handle(ref_table, handle):  # 用稳定 handle 精确解析实体。
    if handle not in ref_table:  # 引用不存在的 handle 应报错。
        raise KeyError(handle)  # 抛出未知引用错误。
    return ref_table[handle]  # 返回精确命中的实体。

target_by_handle = resolve_by_handle(ref_table, "ref://order/a")  # 用 handle 精确引用订单 A。
print("handle 引用目标:", target_by_handle["handle"], target_by_handle["customer"])  # 展示精确命中订单 A。

handle 引用目标: ref://order/a Alice


In [4]:
def ship(order):  # 对指定订单发货。
    return {"shipped_to": order["customer"], "handle": order["handle"]}  # 返回发货对象。

ship_language = ship(target_by_language)  # 基于自然语言指代发货。
ship_handle = ship(target_by_handle)  # 基于 handle 引用发货。
print("自然语言指代发货给:", ship_language["shipped_to"], "(本应是 Alice)")  # 展示指代错误发错人。
print("handle 引用发货给:", ship_handle["shipped_to"], "(正确 Alice)")  # 展示 handle 引用发对人。

自然语言指代发货给: Bob (本应是 Alice)
handle 引用发货给: Alice (正确 Alice)


## 结果解读

自然语言指代漂移到订单 B，把货发给了 Bob；handle 引用精确命中订单 A，发给 Alice。要点：实体产出时由外壳分配稳定 handle 并登记 ref 表，后续一律用 handle 引用，引用不存在应报错。

In [5]:
print("自然语言指代 handle:", target_by_language["handle"])  # 自然语言指代命中的 handle。
print("handle 引用 handle:", target_by_handle["handle"])  # handle 精确引用的 handle。
print("两者是否一致:", target_by_language["handle"] == target_by_handle["handle"])  # 展示指代漂移导致不一致。

自然语言指代 handle: ref://order/b
handle 引用 handle: ref://order/a
两者是否一致: False


In [6]:
assert target_by_handle["handle"] == "ref://order/a"  # handle 引用精确命中订单 A。
assert target_by_handle["customer"] == "Alice"  # handle 引用得到正确客户。
assert target_by_language["handle"] == "ref://order/b"  # 自然语言指代漂移到了订单 B。
assert ship_handle["shipped_to"] == "Alice"  # handle 引用发货给正确客户。
assert ship_language["shipped_to"] == "Bob"  # 自然语言指代发错了客户。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
